In [5]:
# 05_aml_baseline_chrono — Cell 1: setup + verify AML source
import sys
from pathlib import Path

REPO = Path("C:/m1/repo")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import duckdb
import pandas as pd
import numpy as np

from src.config import PATHS, SEED, DUCKDB_FILE, duckdb_connect, Timer

TRAIN_CUTOFF = pd.Timestamp("2022-09-06 13:36:00")

con = duckdb_connect()

print("repo:", PATHS["repo"])
print("work:", PATHS["work"])
print("lake:", PATHS["lake"])
print("DuckDB:", duckdb.__version__)
print("SEED:", SEED)
print("TRAIN_CUTOFF:", TRAIN_CUTOFF)
print("DUCKDB_FILE:", DUCKDB_FILE, "| exists:", DUCKDB_FILE.exists())

WORK_PARQUET = PATHS["work"] / "aml_hismall_clean.parquet"
LAKE_PARQUET = PATHS["lake"] / "aml_hismall_clean.parquet"
print("\nwork parquet:", WORK_PARQUET, "| exists:", WORK_PARQUET.exists())
print("lake parquet:", LAKE_PARQUET, "| exists:", LAKE_PARQUET.exists())

tables = con.execute("SHOW TABLES").fetchdf()
print("\nTables in DUCKDB_FILE:")
print(tables)

if "aml" in tables["name"].values:
    print("\naml schema:")
    print(con.execute("DESCRIBE aml").fetchdf())
    print("\naml row count:", con.execute("SELECT COUNT(*) FROM aml").fetchone()[0])

repo: C:\m1\repo
work: C:\m1\work
lake: D:\m1\lake
DuckDB: 1.5.5
SEED: 42
TRAIN_CUTOFF: 2022-09-06 13:36:00
DUCKDB_FILE: C:\m1\work\m1.duckdb | exists: True

work parquet: C:\m1\work\aml_hismall_clean.parquet | exists: True
lake parquet: D:\m1\lake\aml_hismall_clean.parquet | exists: True

Tables in DUCKDB_FILE:
  name
0  aml

aml schema:
      column_name column_type null   key default extra
0              ts   TIMESTAMP  YES  None    None  None
1       from_bank     VARCHAR  YES  None    None  None
2       from_acct     VARCHAR  YES  None    None  None
3         to_bank     VARCHAR  YES  None    None  None
4         to_acct     VARCHAR  YES  None    None  None
5    amt_received      DOUBLE  YES  None    None  None
6        recv_ccy     VARCHAR  YES  None    None  None
7        amt_paid      DOUBLE  YES  None    None  None
8         pay_ccy     VARCHAR  YES  None    None  None
9      pay_format     VARCHAR  YES  None    None  None
10  is_laundering     TINYINT  YES  None    None  None

In [8]:
# 05_aml_baseline_chrono — Cell 2: FX rate calibration (train-only) + stability check
import numpy as np
import pandas as pd

with Timer("FX: pull train cross-currency pairs"):
    fx_train = con.execute(f"""
        SELECT ts, pay_ccy, recv_ccy, amt_paid, amt_received
        FROM aml
        WHERE ts < TIMESTAMP '{TRAIN_CUTOFF}'
          AND pay_ccy <> recv_ccy
          AND amt_paid > 0 AND amt_received > 0
    """).fetchdf()

print("cross-ccy train rows:", len(fx_train))
print(fx_train[["pay_ccy", "recv_ccy"]].drop_duplicates().shape[0], "distinct ccy pairs")
print("ts range:", fx_train["ts"].min(), "to", fx_train["ts"].max())

ANCHOR_CCY = "US Dollar"

def solve_fx_rates(df):
    """Least squares on log(amt_received/amt_paid) = log(u_pay) - log(u_recv), anchor US Dollar=1."""
    currencies = sorted(set(df["pay_ccy"]) | set(df["recv_ccy"]))
    idx = {c: i for i, c in enumerate(currencies)}
    n = len(currencies)

    y = np.log(df["amt_received"].values / df["amt_paid"].values)
    A = np.zeros((len(df), n))
    for row_i, (p, r) in enumerate(zip(df["pay_ccy"], df["recv_ccy"])):
        A[row_i, idx[p]] += 1.0
        A[row_i, idx[r]] -= 1.0

    assert ANCHOR_CCY in idx, f"{ANCHOR_CCY} not present in currency set"
    anchor_col = idx[ANCHOR_CCY]
    cols = [c for c in range(n) if c != anchor_col]
    A_reduced = A[:, cols]

    log_u_reduced, *_ = np.linalg.lstsq(A_reduced, y, rcond=None)

    log_u = np.zeros(n)
    for k, c in enumerate(cols):
        log_u[c] = log_u_reduced[k]

    return {currencies[i]: float(np.exp(log_u[i])) for i in range(n)}, currencies

rates_full, currencies_full = solve_fx_rates(fx_train)

day1_start = pd.Timestamp("2022-09-01 00:00:00")
day1_end = pd.Timestamp("2022-09-02 00:00:00")
fx_day1 = fx_train[(fx_train["ts"] >= day1_start) & (fx_train["ts"] < day1_end)]
print("\nday-1 cross-ccy rows:", len(fx_day1))

rates_day1, currencies_day1 = solve_fx_rates(fx_day1)

common = sorted(set(currencies_full) & set(currencies_day1))
missing_in_day1 = sorted(set(currencies_full) - set(currencies_day1))

print("\ncurrencies in full train :", len(currencies_full), currencies_full)
print("currencies in day-1      :", len(currencies_day1), currencies_day1)
if missing_in_day1:
    print("MISSING from day-1 (cannot check stability for these):", missing_in_day1)

rows = []
for c in common:
    full_r = rates_full[c]
    d1_r = rates_day1[c]
    pct_diff = abs(d1_r - full_r) / full_r * 100 if full_r != 0 else np.nan
    rows.append({"currency": c, "rate_full_train": full_r, "rate_day1": d1_r, "pct_diff": pct_diff})

stability_df = pd.DataFrame(rows).sort_values("pct_diff", ascending=False)
print("\nFX stability (day-1 vs full-train), sorted by %% diff:")
print(stability_df.to_string(index=False))

max_diff = stability_df["pct_diff"].max()
print(f"\nMax %% diff: {max_diff:.4f}%")
if max_diff < 1.0 and not missing_in_day1:
    print("STABLE -> proceed with train-period cross-currency FX rates (single fixed rate per currency).")
    FX_MODE = "cross_currency"
else:
    print("UNSTABLE or missing coverage -> FALLBACK per #9a: log(amt_paid) + pay_ccy, no cross-currency sums.")
    FX_MODE = "fallback_no_fx"

print("\nFX_MODE =", FX_MODE)

[FX: pull train cross-currency pairs] wall = 0.1 s | RSS now = 0.22 GB | peak RSS = 0.23 GB
cross-ccy train rows: 39749
193 distinct ccy pairs
ts range: 2022-09-01 00:00:00 to 2022-09-06 13:35:00

day-1 cross-ccy rows: 9572

currencies in full train : 15 ['Australian Dollar', 'Bitcoin', 'Brazil Real', 'Canadian Dollar', 'Euro', 'Mexican Peso', 'Ruble', 'Rupee', 'Saudi Riyal', 'Shekel', 'Swiss Franc', 'UK Pound', 'US Dollar', 'Yen', 'Yuan']
currencies in day-1      : 15 ['Australian Dollar', 'Bitcoin', 'Brazil Real', 'Canadian Dollar', 'Euro', 'Mexican Peso', 'Ruble', 'Rupee', 'Saudi Riyal', 'Shekel', 'Swiss Franc', 'UK Pound', 'US Dollar', 'Yen', 'Yuan']

FX stability (day-1 vs full-train), sorted by %% diff:
         currency  rate_full_train    rate_day1  pct_diff
          Bitcoin     11740.741019 11702.313977  0.327297
      Saudi Riyal         0.266354     0.265637  0.269040
     Mexican Peso         0.047394     0.047511  0.246412
            Ruble         0.012860     0.012882  

In [9]:
# 05_aml_baseline_chrono — Cell 3: apply FX rates, create amt_usd columns, persist aml_fx table
import pandas as pd

assert FX_MODE == "cross_currency"
assert set(rates_full.keys()) == {
    'Australian Dollar','Bitcoin','Brazil Real','Canadian Dollar','Euro','Mexican Peso',
    'Ruble','Rupee','Saudi Riyal','Shekel','Swiss Franc','UK Pound','US Dollar','Yen','Yuan'
}, "rate coverage mismatch"

rates_df = pd.DataFrame(
    [{"ccy": c, "rate_to_usd": r} for c, r in rates_full.items()]
)
print(rates_df.sort_values("ccy").to_string(index=False))

con.execute("DROP TABLE IF EXISTS fx_rates")
con.register("rates_df_view", rates_df)
con.execute("CREATE TABLE fx_rates AS SELECT * FROM rates_df_view")

with Timer("build aml_fx (amt_paid_usd, amt_received_usd)"):
    con.execute("""
        DROP TABLE IF EXISTS aml_fx;
        CREATE TABLE aml_fx AS
        SELECT
            a.*,
            a.amt_paid     * fp.rate_to_usd AS amt_paid_usd,
            a.amt_received * fr.rate_to_usd AS amt_received_usd,
            (a.pay_ccy <> a.recv_ccy) AS is_cross_ccy
        FROM aml a
        JOIN fx_rates fp ON a.pay_ccy  = fp.ccy
        JOIN fx_rates fr ON a.recv_ccy = fr.ccy
    """)

n_orig = con.execute("SELECT COUNT(*) FROM aml").fetchone()[0]
n_fx = con.execute("SELECT COUNT(*) FROM aml_fx").fetchone()[0]
print(f"\naml rows: {n_orig} | aml_fx rows: {n_fx} | match: {n_orig == n_fx}")

print("\naml_fx schema:")
print(con.execute("DESCRIBE aml_fx").fetchdf())

print("\nsanity: amt_paid_usd vs amt_received_usd, same-currency rows (should be ~equal, no FX applied):")
print(con.execute("""
    SELECT amt_paid, amt_paid_usd, amt_received, amt_received_usd, pay_ccy, recv_ccy
    FROM aml_fx WHERE pay_ccy = recv_ccy AND pay_ccy = 'US Dollar' LIMIT 3
""").fetchdf())

print("\nsanity: cross-currency row example:")
print(con.execute("""
    SELECT amt_paid, amt_paid_usd, pay_ccy, amt_received, amt_received_usd, recv_ccy
    FROM aml_fx WHERE is_cross_ccy = true LIMIT 3
""").fetchdf())

              ccy  rate_to_usd
Australian Dollar     0.707678
          Bitcoin 11740.741019
      Brazil Real     0.177083
  Canadian Dollar     0.757974
             Euro     1.171777
     Mexican Peso     0.047394
            Ruble     0.012860
            Rupee     0.013616
      Saudi Riyal     0.266354
           Shekel     0.296163
      Swiss Franc     1.092991
         UK Pound     1.291278
        US Dollar     1.000000
              Yen     0.009494
             Yuan     0.149452


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[build aml_fx (amt_paid_usd, amt_received_usd)] wall = 5.5 s | RSS now = 0.76 GB | peak RSS = 1.60 GB

aml rows: 5078345 | aml_fx rows: 5078345 | match: True

aml_fx schema:
         column_name column_type null   key default extra
0                 ts   TIMESTAMP  YES  None    None  None
1          from_bank     VARCHAR  YES  None    None  None
2          from_acct     VARCHAR  YES  None    None  None
3            to_bank     VARCHAR  YES  None    None  None
4            to_acct     VARCHAR  YES  None    None  None
5       amt_received      DOUBLE  YES  None    None  None
6           recv_ccy     VARCHAR  YES  None    None  None
7           amt_paid      DOUBLE  YES  None    None  None
8            pay_ccy     VARCHAR  YES  None    None  None
9         pay_format     VARCHAR  YES  None    None  None
10     is_laundering     TINYINT  YES  None    None  None
11               src     VARCHAR  YES  None    None  None
12               dst     VARCHAR  YES  None    None  None
13      amt_pa

In [10]:
# 05_aml_baseline_chrono — Cell 4a: sender out_/in_ cnt+sum, windows 1h/24h/7d (12 features)
import numpy as np
import pandas as pd

# --- deterministic id, guarded against re-run ---
if "id" not in con.execute("DESCRIBE aml_fx").fetchdf()["column_name"].values:
    with Timer("add deterministic id to aml_fx"):
        con.execute("""
            CREATE OR REPLACE TABLE aml_fx AS
            SELECT ROW_NUMBER() OVER (ORDER BY ts, src, dst, amt_paid) AS id, *
            FROM aml_fx
        """)
print("aml_fx columns:", list(con.execute("SELECT * FROM aml_fx LIMIT 0").fetchdf().columns))

# --- per-(account, ts) rollups, self-loops excluded, then cumulative ---
with Timer("build cum_out, cum_in"):
    con.execute("""
        CREATE OR REPLACE TABLE out_per_minute AS
        SELECT src AS account, ts, COUNT(*) AS cnt, SUM(amt_paid_usd) AS amt_sum
        FROM aml_fx
        WHERE src <> dst
        GROUP BY src, ts
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cum_out AS
        SELECT account, ts,
            SUM(cnt) OVER (PARTITION BY account ORDER BY ts
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_cnt,
            SUM(amt_sum) OVER (PARTITION BY account ORDER BY ts
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_sum
        FROM out_per_minute
    """)
    con.execute("""
        CREATE OR REPLACE TABLE in_per_minute AS
        SELECT dst AS account, ts, COUNT(*) AS cnt, SUM(amt_received_usd) AS amt_sum
        FROM aml_fx
        WHERE src <> dst
        GROUP BY dst, ts
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cum_in AS
        SELECT account, ts,
            SUM(cnt) OVER (PARTITION BY account ORDER BY ts
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_cnt,
            SUM(amt_sum) OVER (PARTITION BY account ORDER BY ts
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_sum
        FROM in_per_minute
    """)

print("cum_out rows:", con.execute("SELECT COUNT(*) FROM cum_out").fetchone()[0])
print("cum_in rows :", con.execute("SELECT COUNT(*) FROM cum_in").fetchone()[0])

# --- sender_features via ASOF joins ---
with Timer("sender_features (ASOF)"):
    con.execute("""
        CREATE OR REPLACE TABLE sender_features AS
        WITH t AS (
            SELECT id, src, ts,
                   ts - INTERVAL '1 minute' AS upper_x,
                   ts - INTERVAL '1 hour'   - INTERVAL '1 minute' AS lower_1h,
                   ts - INTERVAL '24 hours' - INTERVAL '1 minute' AS lower_24h,
                   ts - INTERVAL '7 days'   - INTERVAL '1 minute' AS lower_7d
            FROM aml_fx
        )
        SELECT
            t.id,
            COALESCE(ou.cum_cnt,0)  - COALESCE(ol1.cum_cnt,0)  AS out_cnt_1h,
            COALESCE(ou.cum_cnt,0)  - COALESCE(ol24.cum_cnt,0) AS out_cnt_24h,
            COALESCE(ou.cum_cnt,0)  - COALESCE(ol7.cum_cnt,0)  AS out_cnt_7d,
            COALESCE(ou.cum_sum,0)  - COALESCE(ol1.cum_sum,0)  AS out_sum_usd_1h,
            COALESCE(ou.cum_sum,0)  - COALESCE(ol24.cum_sum,0) AS out_sum_usd_24h,
            COALESCE(ou.cum_sum,0)  - COALESCE(ol7.cum_sum,0)  AS out_sum_usd_7d,
            COALESCE(iu.cum_cnt,0)  - COALESCE(il1.cum_cnt,0)  AS in_cnt_1h,
            COALESCE(iu.cum_cnt,0)  - COALESCE(il24.cum_cnt,0) AS in_cnt_24h,
            COALESCE(iu.cum_cnt,0)  - COALESCE(il7.cum_cnt,0)  AS in_cnt_7d,
            COALESCE(iu.cum_sum,0)  - COALESCE(il1.cum_sum,0)  AS in_sum_usd_1h,
            COALESCE(iu.cum_sum,0)  - COALESCE(il24.cum_sum,0) AS in_sum_usd_24h,
            COALESCE(iu.cum_sum,0)  - COALESCE(il7.cum_sum,0)  AS in_sum_usd_7d
        FROM t
        ASOF LEFT JOIN cum_out ou   ON t.src = ou.account   AND ou.ts   <= t.upper_x
        ASOF LEFT JOIN cum_out ol1  ON t.src = ol1.account  AND ol1.ts  <= t.lower_1h
        ASOF LEFT JOIN cum_out ol24 ON t.src = ol24.account AND ol24.ts <= t.lower_24h
        ASOF LEFT JOIN cum_out ol7  ON t.src = ol7.account  AND ol7.ts  <= t.lower_7d
        ASOF LEFT JOIN cum_in  iu   ON t.src = iu.account   AND iu.ts   <= t.upper_x
        ASOF LEFT JOIN cum_in  il1  ON t.src = il1.account  AND il1.ts  <= t.lower_1h
        ASOF LEFT JOIN cum_in  il24 ON t.src = il24.account AND il24.ts <= t.lower_24h
        ASOF LEFT JOIN cum_in  il7  ON t.src = il7.account  AND il7.ts  <= t.lower_7d
    """)

n_sf = con.execute("SELECT COUNT(*) FROM sender_features").fetchone()[0]
n_aml = con.execute("SELECT COUNT(*) FROM aml_fx").fetchone()[0]
print(f"\nsender_features rows: {n_sf} | aml_fx rows: {n_aml} | match: {n_sf == n_aml}")
print("\nsender_features columns:", list(con.execute("SELECT * FROM sender_features LIMIT 0").fetchdf().columns))
print(con.execute("SELECT * FROM sender_features LIMIT 5").fetchdf())

# --- monotonicity assert ---
bad = con.execute("""
    SELECT COUNT(*) FROM sender_features
    WHERE out_cnt_1h > out_cnt_24h OR out_cnt_24h > out_cnt_7d
       OR in_cnt_1h  > in_cnt_24h  OR in_cnt_24h  > in_cnt_7d
       OR out_sum_usd_1h > out_sum_usd_24h + 1e-9 OR out_sum_usd_24h > out_sum_usd_7d + 1e-9
       OR in_sum_usd_1h  > in_sum_usd_24h  + 1e-9 OR in_sum_usd_24h  > in_sum_usd_7d  + 1e-9
""").fetchone()[0]
print(f"\nmonotonicity violations: {bad}")
assert bad == 0, f"{bad} rows violate monotonicity"

# --- first-minute invariant ---
first_minute = con.execute("SELECT MIN(ts) FROM aml_fx").fetchone()[0]
n_first = con.execute(f"SELECT COUNT(*) FROM aml_fx WHERE ts = TIMESTAMP '{first_minute}'").fetchone()[0]
n_zero = con.execute(f"""
    SELECT COUNT(*) FROM aml_fx a JOIN sender_features sf ON a.id = sf.id
    WHERE a.ts = TIMESTAMP '{first_minute}'
      AND sf.out_cnt_1h = 0 AND sf.out_cnt_24h = 0 AND sf.out_cnt_7d = 0
      AND sf.in_cnt_1h = 0 AND sf.in_cnt_24h = 0 AND sf.in_cnt_7d = 0
""").fetchone()[0]
print(f"\nfirst-minute rows: {n_first} | all-zero sender counts: {n_zero} | match: {n_first == n_zero}")
assert n_first == n_zero, "first-minute invariant FAILED"

# --- brute-force check: stratified sample (first tx + 4 random), self-loops excluded ---
first_tx_id = con.execute("SELECT id FROM aml_fx ORDER BY ts, id LIMIT 1").fetchone()[0]
random_ids = con.execute(f"""
    SELECT id FROM aml_fx USING SAMPLE 4 (bernoulli, 1%) REPEATABLE (42) LIMIT 4
""").fetchdf()["id"].tolist()
sample_ids = [first_tx_id] + random_ids
print("\nsample ids (first=cold-start, rest=random):", sample_ids)

mismatches = []
for tx_id in sample_ids:
    row = con.execute(f"SELECT id, src, ts FROM aml_fx WHERE id = {tx_id}").fetchdf().iloc[0]
    computed = con.execute(f"SELECT * FROM sender_features WHERE id = {tx_id}").fetchdf().iloc[0]

    for label, hours in [("1h", 1), ("24h", 24), ("7d", 168)]:
        bf_out = con.execute(f"""
            SELECT COUNT(*) AS cnt, COALESCE(SUM(amt_paid_usd),0) AS out_sum
            FROM aml_fx
            WHERE src = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '{hours} hours'
              AND ts <  TIMESTAMP '{row.ts}'
        """).fetchdf().iloc[0]
        bf_in = con.execute(f"""
            SELECT COUNT(*) AS cnt, COALESCE(SUM(amt_received_usd),0) AS in_sum
            FROM aml_fx
            WHERE dst = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '{hours} hours'
              AND ts <  TIMESTAMP '{row.ts}'
        """).fetchdf().iloc[0]

        out_cnt_col, out_sum_col = f"out_cnt_{label}", f"out_sum_usd_{label}"
        in_cnt_col, in_sum_col = f"in_cnt_{label}", f"in_sum_usd_{label}"

        ok_out_cnt = bf_out["cnt"] == computed[out_cnt_col]
        ok_out_sum = abs(bf_out["out_sum"] - computed[out_sum_col]) < 1e-6
        ok_in_cnt = bf_in["cnt"] == computed[in_cnt_col]
        ok_in_sum = abs(bf_in["in_sum"] - computed[in_sum_col]) < 1e-6

        if not (ok_out_cnt and ok_out_sum and ok_in_cnt and ok_in_sum):
            mismatches.append((tx_id, label, dict(
                bf_out_cnt=bf_out["cnt"], computed_out_cnt=computed[out_cnt_col],
                bf_out_sum=bf_out["out_sum"], computed_out_sum=computed[out_sum_col],
                bf_in_cnt=bf_in["cnt"], computed_in_cnt=computed[in_cnt_col],
                bf_in_sum=bf_in["in_sum"], computed_in_sum=computed[in_sum_col],
            )))

print(f"\nbrute-force check: {len(sample_ids)} rows x 3 windows = {len(sample_ids)*3} checks")
print("mismatches:", len(mismatches))
for m in mismatches:
    print(m)

assert len(mismatches) == 0, "Cell 4a brute-force verification FAILED"
print("\nCell 4a: 12 features built and verified (self-loops excluded, monotonicity holds).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[add deterministic id to aml_fx] wall = 16.8 s | RSS now = 0.76 GB | peak RSS = 3.28 GB
aml_fx columns: ['id', 'ts', 'from_bank', 'from_acct', 'to_bank', 'to_acct', 'amt_received', 'recv_ccy', 'amt_paid', 'pay_ccy', 'pay_format', 'is_laundering', 'src', 'dst', 'amt_paid_usd', 'amt_received_usd', 'is_cross_ccy']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[build cum_out, cum_in] wall = 8.0 s | RSS now = 1.09 GB | peak RSS = 3.28 GB
cum_out rows: 4075227
cum_in rows : 4364273


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[sender_features (ASOF)] wall = 43.9 s | RSS now = 0.23 GB | peak RSS = 9.06 GB

sender_features rows: 5078345 | aml_fx rows: 5078345 | match: True

sender_features columns: ['id', 'out_cnt_1h', 'out_cnt_24h', 'out_cnt_7d', 'out_sum_usd_1h', 'out_sum_usd_24h', 'out_sum_usd_7d', 'in_cnt_1h', 'in_cnt_24h', 'in_cnt_7d', 'in_sum_usd_1h', 'in_sum_usd_24h', 'in_sum_usd_7d']
        id  out_cnt_1h  out_cnt_24h  out_cnt_7d  out_sum_usd_1h  \
0  3900783         0.0          9.0        50.0        0.000000   
1  3953674         0.0         10.0        49.0        0.000000   
2  3958216         1.0         11.0        50.0    77674.886162   
3  3958544         2.0         12.0        51.0   782537.356652   
4  3960469         3.0         13.0        52.0   817710.060458   

   out_sum_usd_24h  out_sum_usd_7d  in_cnt_1h  in_cnt_24h  in_cnt_7d  \
0     1.737340e+06    8.690317e+06        0.0         3.0       20.0   
1     1.737572e+06    8.689433e+06        1.0         4.0       21.0   
2     1.81

ParserException: Parser Error: syntax error at or near "%"

LINE 2:     SELECT id FROM aml_fx USING SAMPLE 4 (bernoulli, 1%) REPEATABLE (42) LIMIT 4
                                                              ^

In [11]:
# brute-force check: stratified sample (first tx + 4 random), self-loops excluded
import numpy as np
np.random.seed(42)

total = con.execute("SELECT COUNT(*) FROM aml_fx").fetchone()[0]
sample_offsets = sorted(np.random.choice(total, size=4, replace=False).tolist())
random_ids = [
    con.execute(f"SELECT id FROM aml_fx ORDER BY id LIMIT 1 OFFSET {o}").fetchone()[0]
    for o in sample_offsets
]
print("sample offsets:", sample_offsets)

first_tx_id = con.execute("SELECT id FROM aml_fx ORDER BY ts, id LIMIT 1").fetchone()[0]
sample_ids = [first_tx_id] + random_ids
print("sample ids (first=cold-start, rest=random):", sample_ids)

mismatches = []
for tx_id in sample_ids:
    row = con.execute(f"SELECT id, src, ts FROM aml_fx WHERE id = {tx_id}").fetchdf().iloc[0]
    computed = con.execute(f"SELECT * FROM sender_features WHERE id = {tx_id}").fetchdf().iloc[0]

    for label, hours in [("1h", 1), ("24h", 24), ("7d", 168)]:
        bf_out = con.execute(f"""
            SELECT COUNT(*) AS cnt, COALESCE(SUM(amt_paid_usd),0) AS out_sum
            FROM aml_fx
            WHERE src = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '{hours} hours'
              AND ts <  TIMESTAMP '{row.ts}'
        """).fetchdf().iloc[0]
        bf_in = con.execute(f"""
            SELECT COUNT(*) AS cnt, COALESCE(SUM(amt_received_usd),0) AS in_sum
            FROM aml_fx
            WHERE dst = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '{hours} hours'
              AND ts <  TIMESTAMP '{row.ts}'
        """).fetchdf().iloc[0]

        out_cnt_col, out_sum_col = f"out_cnt_{label}", f"out_sum_usd_{label}"
        in_cnt_col, in_sum_col = f"in_cnt_{label}", f"in_sum_usd_{label}"

        ok_out_cnt = bf_out["cnt"] == computed[out_cnt_col]
        ok_out_sum = abs(bf_out["out_sum"] - computed[out_sum_col]) < 1e-6
        ok_in_cnt = bf_in["cnt"] == computed[in_cnt_col]
        ok_in_sum = abs(bf_in["in_sum"] - computed[in_sum_col]) < 1e-6

        if not (ok_out_cnt and ok_out_sum and ok_in_cnt and ok_in_sum):
            mismatches.append((tx_id, label, dict(
                bf_out_cnt=bf_out["cnt"], computed_out_cnt=computed[out_cnt_col],
                bf_out_sum=bf_out["out_sum"], computed_out_sum=computed[out_sum_col],
                bf_in_cnt=bf_in["cnt"], computed_in_cnt=computed[in_cnt_col],
                bf_in_sum=bf_in["in_sum"], computed_in_sum=computed[in_sum_col],
            )))

print(f"\nbrute-force check: {len(sample_ids)} rows x 3 windows = {len(sample_ids)*3} checks")
print("mismatches:", len(mismatches))
for m in mismatches:
    print(m)

assert len(mismatches) == 0, "Cell 4a brute-force verification FAILED"
print("\nCell 4a: 12 features built and verified (self-loops excluded, monotonicity holds).")

sample offsets: [298872, 405190, 746726, 1388703]
sample ids (first=cold-start, rest=random): [1, 298873, 405191, 746727, 1388704]

brute-force check: 5 rows x 3 windows = 15 checks
mismatches: 0

Cell 4a: 12 features built and verified (self-loops excluded, monotonicity holds).


In [12]:
import gc
gc.collect()

355

In [13]:
from pathlib import Path

TEXT = """

## DECISION #9b — mins_since_last_inflow is all-time, not 24h-capped (2026-09-27, before Cell 4b)
- #9 point 4 lists mins_since_last_inflow under the Sender 24h block, but it carries
  no _24h suffix; semantics: minutes since the most recent prior inflow (dst = account),
  at any time before t.ts, not capped to 24h.
- Reason: capping removes information the tree can use directly (it can split on any
  threshold, e.g. <=60, <=1440); the AML-relevant recency window is learned, not imposed.
  pass_through_24h and amt_over_inflow_24h remain windowed (they are ratios against
  24h sums, not raw recency).
- NULL if the account has no prior inflow (cold start).
- Implementation: MAX(prev_ts) over rows where dst = account AND prev_ts <= t.ts - 1 minute
  (same-minute and self-loop inflows excluded, consistent with #9's same-minute rule).
- Distribution note: early-train rows are bounded by dataset start (not full history);
  this is the expanding-window limitation already noted in #9 point 11, not leakage.
"""

p = Path("../reports/decisions.md")
d = p.read_text(encoding="utf-8")
assert "DECISION #9a" in d, "register #9a first"
assert "DECISION #9b" not in d, "#9b already present"
with open(p, "a", encoding="utf-8") as f:
    f.write(TEXT)
print("written")

written


In [17]:
# 05_aml_baseline_chrono — Cell 4b-1: cheap derived features
import numpy as np
import pandas as pd

with Timer("mins_since_last_inflow (ASOF, all-time)"):
    con.execute("""
        CREATE OR REPLACE TABLE last_inflow AS
        WITH t AS (SELECT id, src, ts FROM aml_fx)
        SELECT
            t.id,
            DATEDIFF('minute', h.last_in_ts, t.ts) AS mins_since_last_inflow
        FROM t
        ASOF LEFT JOIN (
            SELECT dst AS account, ts AS last_in_ts
            FROM aml_fx WHERE src <> dst
        ) h
        ON t.src = h.account AND h.last_in_ts <= t.ts - INTERVAL '1 minute'
    """)
    print("last_inflow rows:", con.execute("SELECT COUNT(*) FROM last_inflow").fetchone()[0])

with Timer("sender_derived_cheap"):
    con.execute("""
        CREATE OR REPLACE TABLE sender_derived_cheap AS
        SELECT
            sf.id,
            CASE WHEN sf.in_sum_usd_24h = 0 THEN NULL
                 ELSE sf.out_sum_usd_24h / sf.in_sum_usd_24h END AS pass_through_24h,
            li.mins_since_last_inflow,
            CASE WHEN sf.in_sum_usd_24h = 0 THEN NULL
                 ELSE a.amt_paid_usd / sf.in_sum_usd_24h END AS amt_over_inflow_24h
        FROM sender_features sf
        JOIN aml_fx a ON a.id = sf.id
        JOIN last_inflow li ON li.id = sf.id
    """)
    print("sender_derived_cheap rows:", con.execute("SELECT COUNT(*) FROM sender_derived_cheap").fetchone()[0])
    print(con.execute("SELECT * FROM sender_derived_cheap LIMIT 5").fetchdf())

n_null_pt = con.execute("SELECT COUNT(*) FROM sender_derived_cheap WHERE pass_through_24h IS NULL").fetchone()[0]
n_null_aoi = con.execute("SELECT COUNT(*) FROM sender_derived_cheap WHERE amt_over_inflow_24h IS NULL").fetchone()[0]
n_zero_insum = con.execute("SELECT COUNT(*) FROM sender_features WHERE in_sum_usd_24h = 0").fetchone()[0]
print(f"NULL pass_through: {n_null_pt} | NULL amt_over_inflow: {n_null_aoi} | zero in_sum: {n_zero_insum}")
assert n_null_pt == n_zero_insum and n_null_aoi == n_zero_insum
print("\nCell 4b-1: 3 features built. Total: 15/34.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

last_inflow rows: 5078345
[mins_since_last_inflow (ASOF, all-time)] wall = 5.2 s | RSS now = 0.34 GB | peak RSS = 9.39 GB
sender_derived_cheap rows: 5078345
        id  pass_through_24h  mins_since_last_inflow  amt_over_inflow_24h
0  3900783        450.952224                    1020             0.060413
1  3953674        353.006517                       3            15.780489
2  3958216        237.900392                      12            92.376983
3  3958544        330.277376                      13             4.609620
4  3960469        331.630700                       5             6.420832
[sender_derived_cheap] wall = 1.0 s | RSS now = 0.85 GB | peak RSS = 9.39 GB
NULL pass_through: 2395001 | NULL amt_over_inflow: 2395001 | zero in_sum: 2395001

Cell 4b-1: 3 features built. Total: 15/34.


In [20]:
# 05_aml_baseline_chrono — Cell 4b-2: new-counterparty rate 24h (DECISION #9c), 2 features
# Single LAG per (src,dst), projected to both directions. Supersedes the two-LAG version.

with Timer("pair_events: LAG once, tie-break by id"):
    con.execute("""
        CREATE OR REPLACE TABLE pair_events AS
        SELECT id, src, dst, ts,
               LAG(ts) OVER (PARTITION BY src, dst ORDER BY ts, id) AS prev_pair_ts
        FROM aml_fx
        WHERE src <> dst
    """)
    con.execute("""
        CREATE OR REPLACE TABLE pair_events_flagged AS
        SELECT id, src, dst, ts,
               (prev_pair_ts IS NULL OR ts - prev_pair_ts > INTERVAL '24 hours') AS is_new_pair
        FROM pair_events
    """)

n_total = con.execute("SELECT COUNT(*) FROM pair_events_flagged").fetchone()[0]
n_new = con.execute("SELECT COUNT(*) FROM pair_events_flagged WHERE is_new_pair").fetchone()[0]
print(f"pair_events: {n_total:,} rows | new: {n_new:,} ({100*n_new/n_total:.2f}%)")

con.execute("DROP TABLE pair_events")

with Timer("cumulative new-pair + ASOF"):
    con.execute("""
        CREATE OR REPLACE TABLE out_new_per_minute AS
        SELECT src AS account, ts, COUNT(*) FILTER (WHERE is_new_pair) AS new_cnt
        FROM pair_events_flagged GROUP BY src, ts
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cum_out_new AS
        SELECT account, ts,
               SUM(new_cnt) OVER (PARTITION BY account ORDER BY ts
                                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_new
        FROM out_new_per_minute
    """)
    con.execute("""
        CREATE OR REPLACE TABLE in_new_per_minute AS
        SELECT dst AS account, ts, COUNT(*) FILTER (WHERE is_new_pair) AS new_cnt
        FROM pair_events_flagged GROUP BY dst, ts
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cum_in_new AS
        SELECT account, ts,
               SUM(new_cnt) OVER (PARTITION BY account ORDER BY ts
                                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_new
        FROM in_new_per_minute
    """)
    con.execute("""
        CREATE OR REPLACE TABLE sender_new_counterparty_24h AS
        WITH t AS (
            SELECT id, src, ts,
                   ts - INTERVAL '1 minute' AS upper_x,
                   ts - INTERVAL '24 hours' - INTERVAL '1 minute' AS lower_24h
            FROM aml_fx
        )
        SELECT
            t.id,
            COALESCE(ou.cum_new,0) - COALESCE(ol.cum_new,0) AS n_new_dst_24h,
            COALESCE(iu.cum_new,0) - COALESCE(il.cum_new,0) AS n_new_senders_in_24h
        FROM t
        ASOF LEFT JOIN cum_out_new ou ON t.src = ou.account AND ou.ts <= t.upper_x
        ASOF LEFT JOIN cum_out_new ol ON t.src = ol.account AND ol.ts <= t.lower_24h
        ASOF LEFT JOIN cum_in_new  iu ON t.src = iu.account AND iu.ts <= t.upper_x
        ASOF LEFT JOIN cum_in_new  il ON t.src = il.account AND il.ts <= t.lower_24h
    """)

n_snc = con.execute("SELECT COUNT(*) FROM sender_new_counterparty_24h").fetchone()[0]
assert n_snc == n_aml
print(f"sender_new_counterparty_24h rows: {n_snc} | match: {n_snc == n_aml}")
print(con.execute("SELECT * FROM sender_new_counterparty_24h LIMIT 5").fetchdf())

bad_out = con.execute("""
    SELECT COUNT(*) FROM sender_new_counterparty_24h snc
    JOIN sender_features sf ON sf.id = snc.id
    WHERE snc.n_new_dst_24h > sf.out_cnt_24h
""").fetchone()[0]
bad_in = con.execute("""
    SELECT COUNT(*) FROM sender_new_counterparty_24h snc
    JOIN sender_features sf ON sf.id = snc.id
    WHERE snc.n_new_senders_in_24h > sf.in_cnt_24h
""").fetchone()[0]
print(f"n_new_dst > out_cnt violations: {bad_out} | n_new_senders_in > in_cnt: {bad_in}")
assert bad_out == 0 and bad_in == 0

mismatches_4b2 = []
for tx_id in sample_ids:
    row = con.execute(f"SELECT id, src, ts FROM aml_fx WHERE id = {tx_id}").fetchdf().iloc[0]
    computed = con.execute(f"SELECT * FROM sender_new_counterparty_24h WHERE id = {tx_id}").fetchdf().iloc[0]

    bf_out_new = con.execute(f"""
        WITH window_tx AS (
            SELECT dst, ts FROM aml_fx
            WHERE src = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '24 hours' AND ts < TIMESTAMP '{row.ts}'
        ), with_prev AS (
            SELECT w.dst, w.ts,
                (SELECT MAX(ts) FROM aml_fx p WHERE p.src = '{row.src}' AND p.dst = w.dst AND p.ts < w.ts) AS prev_ts
            FROM window_tx w
        )
        SELECT COUNT(*) AS cnt FROM with_prev
        WHERE prev_ts IS NULL OR ts - prev_ts > INTERVAL '24 hours'
    """).fetchone()[0]
    bf_in_new = con.execute(f"""
        WITH window_tx AS (
            SELECT src, ts FROM aml_fx
            WHERE dst = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '24 hours' AND ts < TIMESTAMP '{row.ts}'
        ), with_prev AS (
            SELECT w.src, w.ts,
                (SELECT MAX(ts) FROM aml_fx p WHERE p.dst = '{row.src}' AND p.src = w.src AND p.ts < w.ts) AS prev_ts
            FROM window_tx w
        )
        SELECT COUNT(*) AS cnt FROM with_prev
        WHERE prev_ts IS NULL OR ts - prev_ts > INTERVAL '24 hours'
    """).fetchone()[0]

    ok_out = bf_out_new == computed["n_new_dst_24h"]
    ok_in  = bf_in_new  == computed["n_new_senders_in_24h"]
    if not (ok_out and ok_in):
        mismatches_4b2.append((tx_id, dict(
            bf_out=bf_out_new, db_out=int(computed["n_new_dst_24h"]),
            bf_in=bf_in_new,  db_in=int(computed["n_new_senders_in_24h"]))))

print(f"\nCell 4b-2 brute-force: {len(sample_ids)} rows | mismatches: {len(mismatches_4b2)}")
for m in mismatches_4b2:
    print(m)
assert len(mismatches_4b2) == 0

print("\nCell 4b-2 done: 2 features. Total: 17/34.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[pair_events: LAG once, tie-break by id] wall = 7.7 s | RSS now = 1.15 GB | peak RSS = 9.39 GB
pair_events: 4,487,133 rows | new: 1,613,573 (35.96%)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[cumulative new-pair + ASOF] wall = 44.7 s | RSS now = 1.18 GB | peak RSS = 9.39 GB
sender_new_counterparty_24h rows: 5078345 | match: True
        id  n_new_dst_24h  n_new_senders_in_24h
0  1531800            2.0                   0.0
1  1538542            2.0                   0.0
2  1539042            2.0                   0.0
3  1540567            2.0                   0.0
4  1549307            2.0                   0.0
n_new_dst > out_cnt violations: 0 | n_new_senders_in > in_cnt: 0

Cell 4b-2 brute-force: 5 rows | mismatches: 0

Cell 4b-2 done: 2 features. Total: 17/34.


In [16]:
from pathlib import Path

TEXT = """

## DECISION #9c — fan-out/fan-in 24h redefined as new-counterparty count, not true distinct (2026-09-27, after OOM on self-join)
- Original #9 point 4: n_unique_dst_out (fan-out), n_unique_src_in (fan-in) as
  COUNT(DISTINCT counterparty) over 24h.
- Problem: a self-join within the 24h window to compute COUNT(DISTINCT) cross-multiplies
  matched outflow rows by matched inflow rows per t (double LEFT JOIN), producing fan-out
  proportional to activity^2 for high-activity accounts; this exhausted 18.6GB DuckDB temp
  space (OutOfMemoryException) after ~17 minutes.
- A correlated-subquery alternative was cost-estimated at ~3 hours (5M rows x 2 subqueries
  x ~1ms each even with indexes), exceeding the #9 point 10 time-box.
- Decision: redefine n_unique_dst_out_24h -> new_counterparty_out_24h, and
  src_n_senders_in_24h -> new_counterparty_in_24h. Computed via LAG() per (src,dst) pair:
  a transaction counts as "new" if its (src,dst) pair had no prior transaction, or its
  prior transaction was more than 24h before t.ts. Feature = count of such "new" pairs
  for the account within [t.ts-24h, t.ts).
- This is not identical to true distinct-counterparty-count (it does not decrement when
  a previously-new pair ages out of the window in the same way; it counts newly-appearing
  pairs, not currently-distinct pairs in-window). It is a standard AML signal in its own
  right ("new counterparty rate") and is computable via a single windowed LAG, no self-join.
- Renamed feature names logged here supersede #9 point 4's n_unique_dst_out / n_unique_src_in.
"""

p = Path("../reports/decisions.md")
d = p.read_text(encoding="utf-8")
assert "DECISION #9b" in d, "register #9b first"
assert "DECISION #9c" not in d, "#9c already present"
with open(p, "a", encoding="utf-8") as f:
    f.write(TEXT)
print("written")

written


In [22]:
from pathlib import Path

TEXT = """

## DECISION #9d — receiver naming, receiver new-counterparty, cold-start sums, FX estimator (2026-09-27, before Cell 4c)
1. Naming: receiver (dst-side) features carry a dst_ prefix; sender features keep the
   #9/#9a/#9c names. Receiver names: dst_in_cnt_1h/24h/7d, dst_in_sum_usd_1h/24h/7d,
   dst_out_cnt_24h, dst_new_counterparty_in_24h (8).
2. Receiver fan-in follows #9c: dst_n_senders_in_24h (#9a) -> dst_new_counterparty_in_24h
   = number of transfers into dst within [t.ts-24h, t.ts) whose (src,dst) pair had no
   earlier transfer, or whose previous transfer was more than 24h earlier. Same
   single-LAG pair table as Cell 4b-2 (tie-break by id).
3. Column mapping for Cell 4b-2 output, applied when features are assembled (4f):
   n_new_dst_24h -> new_counterparty_out_24h; n_new_senders_in_24h ->
   new_counterparty_in_24h. Definitions unchanged.
4. Cold start (#3/#4, #9 point 5): a windowed sum is NULL when the account has no
   earlier non-self-loop event in that direction at all (no history); it is 0 when
   history exists but the window is empty. The distinction applies to sums only;
   counts are 0 in both cases. Receiver sums: implemented in 4c by ASOF NULL
   propagation (no earlier row in cum_in -> NULL). Sender sums: Cell 4a writes 0, so
   4a and 4c are inconsistent until 4f. In 4f the "no history" flag for sender sums is
   recomputed with the same ASOF-no-match logic against cum_out / cum_in; it is NOT
   inferred from sender_features values (a 0 there cannot distinguish the two cases).
5. FX estimator: Cell 2 fitted least squares on log(amt_received/amt_paid) of every
   train cross-currency row, not on per-pair medians as #9 point 2 states. Rule fixed
   before the check: rates from per-pair medians are computed; if every currency
   differs by < 1% from the fitted rates (same threshold as the #9a day-1 check), the
   fitted rates are kept and the measured divergence is logged in pilot_log.md (this
   decision text is not edited afterwards). Otherwise stop and review before any
   further feature code. Switching estimators would require re-running Cells 2, 4a,
   4b-1, 4b-2 and all their checks.
6. Final naming at assembly (Cell 4f): sender features gain a src_ prefix; receiver
   features keep dst_. Applied via SELECT aliases in 4f together with the point-3
   mapping; Cells 4a/4b-1/4b-2 are not re-run.
7. dst_out_cnt_24h is the receiver's own outbound activity (dst acting as sender of
   other transfers), a deliberate mixed-perspective feature from #9 point 4.
8. Verification note: the Cell 4b-2 brute-force ordered "previous pair event" by ts
   only, while the LAG orders by (ts, id). The 5 sample rows did not hit a same-minute
   repeat of a pair, so it passed; the feature is correct, the check was weaker than
   the code. The 4c and 4f brute-force checks order by (ts, id), matching the reference.
"""

p = Path("../reports/decisions.md")
d = p.read_text(encoding="utf-8")
assert "DECISION #9c" in d, "register #9c first"
assert "DECISION #9d" not in d, "#9d already present"
with open(p, "a", encoding="utf-8") as f:
    f.write(TEXT)
print("written")

written


In [23]:
# 05_aml_baseline_chrono — Cell 4c-audit: same-minute repeats of a (src,dst) pair (#9d point 8)
n_groups, n_rows = con.execute("""
    SELECT COUNT(*), COALESCE(SUM(c), 0) FROM (
        SELECT src, dst, ts, COUNT(*) AS c FROM aml_fx
        WHERE src <> dst GROUP BY src, dst, ts HAVING COUNT(*) > 1)
""").fetchone()
n_pe = con.execute("SELECT COUNT(*) FROM aml_fx WHERE src <> dst").fetchone()[0]
print(f"(src,dst,minute) groups with >1 transfer: {n_groups:,} | rows in them: {n_rows:,} "
      f"({100*n_rows/n_pe:.2f}% of {n_pe:,} non-self-loop rows)")

(src,dst,minute) groups with >1 transfer: 96,324 | rows in them: 193,725 (4.32% of 4,487,133 non-self-loop rows)


In [24]:
# 05_aml_baseline_chrono — Cell 4c-0: FX estimator check, per-pair medians vs fitted rates (#9d point 5)
import pandas as pd

assert "fx_train" in globals() and "solve_fx_rates" in globals(), "re-run Cell 2 first"

pair_med = (fx_train.assign(ratio=fx_train["amt_received"] / fx_train["amt_paid"])
            .groupby(["pay_ccy", "recv_ccy"], as_index=False)["ratio"].median())
print("currency pairs:", len(pair_med))

# solve_fx_rates reads log(amt_received / amt_paid); with amt_paid = 1 this is log(median ratio)
pair_med = pair_med.rename(columns={"ratio": "amt_received"}).assign(amt_paid=1.0)
rates_med, _ = solve_fx_rates(pair_med)

cmp = pd.DataFrame({"fitted": pd.Series(rates_full), "median_based": pd.Series(rates_med)})
cmp["pct_diff"] = (cmp["median_based"] - cmp["fitted"]).abs() / cmp["fitted"] * 100
print(cmp.sort_values("pct_diff", ascending=False).to_string())

max_d = cmp["pct_diff"].max()
print(f"\nmax % diff (median-based vs fitted): {max_d:.4f}%")
assert max_d < 1.0, "median-based rates differ by >= 1% -> STOP, review per #9d point 5"
print("Fitted rates kept (#9d point 5). aml_fx unchanged.")

currency pairs: 193
                         fitted  median_based  pct_diff
Bitcoin            11740.741019  11830.793703  0.767010
Shekel                 0.296163      0.293894  0.766263
Mexican Peso           0.047394      0.047611  0.457381
Yuan                   0.149452      0.149685  0.155795
Saudi Riyal            0.266354      0.266706  0.132195
Yen                    0.009494      0.009488  0.059461
Rupee                  0.013616      0.013624  0.057507
Brazil Real            0.177083      0.177179  0.054213
Euro                   1.171777      1.172303  0.044907
Australian Dollar      0.707678      0.707867  0.026591
UK Pound               1.291278      1.291520  0.018732
Ruble                  0.012860      0.012858  0.014691
Swiss Franc            1.092991      1.092942  0.004528
Canadian Dollar        0.757974      0.757998  0.003206
US Dollar              1.000000      1.000000  0.000000

max % diff (median-based vs fitted): 0.7670%
Fitted rates kept (#9d point 5). aml_f

In [25]:
# 05_aml_baseline_chrono — Cell 4c-1: receiver-side features (DECISION #9, #9a, #9c, #9d), 8 features
# Reuses cum_in, cum_out (4a) and cum_in_new (4b-2); ASOF joins on t.dst instead of t.src.
# Sums are NULL when dst has no earlier inflow at all (#9d point 4).

with Timer("receiver_features (ASOF on dst)"):
    con.execute("""
        CREATE OR REPLACE TABLE receiver_features AS
        WITH t AS (
            SELECT id, dst, ts,
                   ts - INTERVAL '1 minute' AS upper_x,
                   ts - INTERVAL '1 hour'   - INTERVAL '1 minute' AS lower_1h,
                   ts - INTERVAL '24 hours' - INTERVAL '1 minute' AS lower_24h,
                   ts - INTERVAL '7 days'   - INTERVAL '1 minute' AS lower_7d
            FROM aml_fx
        )
        SELECT
            t.id,
            COALESCE(iu.cum_cnt,0) - COALESCE(il1.cum_cnt,0)  AS dst_in_cnt_1h,
            COALESCE(iu.cum_cnt,0) - COALESCE(il24.cum_cnt,0) AS dst_in_cnt_24h,
            COALESCE(iu.cum_cnt,0) - COALESCE(il7.cum_cnt,0)  AS dst_in_cnt_7d,
            iu.cum_sum - COALESCE(il1.cum_sum,0)  AS dst_in_sum_usd_1h,
            iu.cum_sum - COALESCE(il24.cum_sum,0) AS dst_in_sum_usd_24h,
            iu.cum_sum - COALESCE(il7.cum_sum,0)  AS dst_in_sum_usd_7d,
            COALESCE(ou.cum_cnt,0) - COALESCE(ol24.cum_cnt,0) AS dst_out_cnt_24h,
            COALESCE(nu.cum_new,0) - COALESCE(nl.cum_new,0)   AS dst_new_counterparty_in_24h
        FROM t
        ASOF LEFT JOIN cum_in     iu   ON t.dst = iu.account   AND iu.ts   <= t.upper_x
        ASOF LEFT JOIN cum_in     il1  ON t.dst = il1.account  AND il1.ts  <= t.lower_1h
        ASOF LEFT JOIN cum_in     il24 ON t.dst = il24.account AND il24.ts <= t.lower_24h
        ASOF LEFT JOIN cum_in     il7  ON t.dst = il7.account  AND il7.ts  <= t.lower_7d
        ASOF LEFT JOIN cum_out    ou   ON t.dst = ou.account   AND ou.ts   <= t.upper_x
        ASOF LEFT JOIN cum_out    ol24 ON t.dst = ol24.account AND ol24.ts <= t.lower_24h
        ASOF LEFT JOIN cum_in_new nu   ON t.dst = nu.account   AND nu.ts   <= t.upper_x
        ASOF LEFT JOIN cum_in_new nl   ON t.dst = nl.account   AND nl.ts   <= t.lower_24h
    """)

n_rf = con.execute("SELECT COUNT(*) FROM receiver_features").fetchone()[0]
print(f"receiver_features rows: {n_rf} | aml_fx rows: {n_aml} | match: {n_rf == n_aml}")
assert n_rf == n_aml
print(con.execute("SELECT * FROM receiver_features LIMIT 5").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[receiver_features (ASOF on dst)] wall = 201.3 s | RSS now = 0.28 GB | peak RSS = 9.39 GB
receiver_features rows: 5078345 | aml_fx rows: 5078345 | match: True
        id  dst_in_cnt_1h  dst_in_cnt_24h  dst_in_cnt_7d  dst_in_sum_usd_1h  \
0  2360210            0.0             0.0           10.0           0.000000   
1  2360209            0.0             0.0           10.0           0.000000   
2  2362848            2.0             2.0           12.0        5581.544644   
3  2364744            3.0             3.0           13.0       10104.800548   
4  3017363            0.0             0.0           14.0           0.000000   

   dst_in_sum_usd_24h  dst_in_sum_usd_7d  dst_out_cnt_24h  \
0            0.000000       45140.362470              0.0   
1            0.000000       45140.362470              0.0   
2         5581.544644       50721.907114              0.0   
3        10104.800548       55245.163019              0.0   
4            0.000000       59633.019242              7.0   


In [26]:
# 05_aml_baseline_chrono — Cell 4c-2: invariants for receiver_features

bad_mono = con.execute("""
    SELECT COUNT(*) FROM receiver_features
    WHERE dst_in_cnt_1h > dst_in_cnt_24h OR dst_in_cnt_24h > dst_in_cnt_7d
       OR dst_in_sum_usd_1h > dst_in_sum_usd_24h + 1e-6 OR dst_in_sum_usd_24h > dst_in_sum_usd_7d + 1e-6
""").fetchone()[0]
bad_new = con.execute("""
    SELECT COUNT(*) FROM receiver_features WHERE dst_new_counterparty_in_24h > dst_in_cnt_24h
""").fetchone()[0]
bad_null = con.execute("""
    SELECT COUNT(*) FROM receiver_features
    WHERE (dst_in_sum_usd_1h IS NULL) <> (dst_in_sum_usd_7d IS NULL)
       OR (dst_in_sum_usd_24h IS NULL) <> (dst_in_sum_usd_7d IS NULL)
       OR (dst_in_sum_usd_7d IS NULL AND dst_in_cnt_7d > 0)
""").fetchone()[0]
bad_cons = con.execute("""
    SELECT COUNT(*) FROM receiver_features
    WHERE (dst_in_cnt_7d = 0 AND dst_in_sum_usd_7d IS NOT NULL AND dst_in_sum_usd_7d <> 0)
       OR (dst_in_cnt_7d > 0 AND (dst_in_sum_usd_7d IS NULL OR dst_in_sum_usd_7d = 0))
""").fetchone()[0]
n_null = con.execute("SELECT COUNT(*) FROM receiver_features WHERE dst_in_sum_usd_7d IS NULL").fetchone()[0]
print(f"monotonicity: {bad_mono} | new > in_cnt_24h: {bad_new} | NULL-pattern: {bad_null} | count<->sum: {bad_cons}")
print(f"cold-start NULL sums: {n_null:,} ({100*n_null/n_aml:.2f}% of rows)")
assert bad_mono == 0 and bad_new == 0 and bad_null == 0 and bad_cons == 0

first_minute = con.execute("SELECT MIN(ts) FROM aml_fx").fetchone()[0]
n_first, n_ok = con.execute(f"""
    SELECT COUNT(*),
           COUNT(*) FILTER (WHERE rf.dst_in_cnt_7d = 0 AND rf.dst_out_cnt_24h = 0
                              AND rf.dst_new_counterparty_in_24h = 0 AND rf.dst_in_sum_usd_7d IS NULL)
    FROM aml_fx a JOIN receiver_features rf ON a.id = rf.id
    WHERE a.ts = TIMESTAMP '{first_minute}'
""").fetchone()
print(f"first-minute rows: {n_first} | zero counts + NULL sums: {n_ok} | match: {n_first == n_ok}")
assert n_first == n_ok

monotonicity: 0 | new > in_cnt_24h: 0 | NULL-pattern: 0 | count<->sum: 0
cold-start NULL sums: 713,911 (14.06% of rows)
first-minute rows: 10977 | zero counts + NULL sums: 10977 | match: True


In [27]:
# 05_aml_baseline_chrono — Cell 4c-3: brute-force check on sample_ids (first tx + 4 random)
import pandas as pd

def close(a, b):
    return abs(a - b) <= 1e-6 * max(1.0, abs(b))

mismatches_4c = []
for tx_id in sample_ids:
    dst, ts = con.execute(f"SELECT dst, ts FROM aml_fx WHERE id = {tx_id}").fetchone()
    c = con.execute(f"SELECT * FROM receiver_features WHERE id = {tx_id}").fetchdf().iloc[0]
    base = f"FROM aml_fx WHERE dst = '{dst}' AND src <> dst AND ts < TIMESTAMP '{ts}'"
    n_hist = con.execute(f"SELECT COUNT(*) {base}").fetchone()[0]

    for label, h in [("1h", 1), ("24h", 24), ("7d", 168)]:
        cnt, s = con.execute(f"SELECT COUNT(*), SUM(amt_received_usd) {base} "
                             f"AND ts >= TIMESTAMP '{ts}' - INTERVAL '{h} hours'").fetchone()
        got = c[f"dst_in_sum_usd_{label}"]
        s_val = 0.0 if s is None else float(s)
        ok_sum = pd.isna(got) if n_hist == 0 else (not pd.isna(got) and close(got, s_val))
        if cnt != c[f"dst_in_cnt_{label}"] or not ok_sum:
            mismatches_4c.append((tx_id, label, cnt, c[f"dst_in_cnt_{label}"], s, got))

    out24 = con.execute(f"""SELECT COUNT(*) FROM aml_fx WHERE src = '{dst}' AND src <> dst
        AND ts >= TIMESTAMP '{ts}' - INTERVAL '24 hours' AND ts < TIMESTAMP '{ts}'""").fetchone()[0]
    new24 = con.execute(f"""
        WITH w AS (SELECT id, src, ts FROM aml_fx WHERE dst = '{dst}' AND src <> dst
                   AND ts >= TIMESTAMP '{ts}' - INTERVAL '24 hours' AND ts < TIMESTAMP '{ts}')
        SELECT COUNT(*) FROM w
        WHERE NOT EXISTS (SELECT 1 FROM aml_fx p WHERE p.dst = '{dst}' AND p.src = w.src
                          AND (p.ts < w.ts OR (p.ts = w.ts AND p.id < w.id))
                          AND p.ts >= w.ts - INTERVAL '24 hours')""").fetchone()[0]
    if out24 != c["dst_out_cnt_24h"]:
        mismatches_4c.append((tx_id, "out_24h", out24, c["dst_out_cnt_24h"]))
    if new24 != c["dst_new_counterparty_in_24h"]:
        mismatches_4c.append((tx_id, "new_24h", new24, c["dst_new_counterparty_in_24h"]))

print(f"Cell 4c brute-force: {len(sample_ids)} rows | mismatches: {len(mismatches_4c)}")
for m in mismatches_4c:
    print(m)
assert len(mismatches_4c) == 0
print("\nCell 4c done: 8 receiver features. Total: 25/34.")

Cell 4c brute-force: 5 rows | mismatches: 0

Cell 4c done: 8 receiver features. Total: 25/34.


In [28]:
from pathlib import Path

assert len(mismatches_4c) == 0 and "max_d" in globals() and "n_groups" in globals()
TEXT = f"""
- Note (2026-09-27): DECISION #9c cell executed at In[16]; the successful Cell 4b-2 run is In[20] (saved execution_count in 05_aml_baseline_chrono.ipynb). #9c was registered before the final 4b-2 run; its position below 4b-2 in the notebook is layout only.
- Note (2026-09-27): #9d registered before Cell 4c. FX estimator check: max divergence between fitted and per-pair-median rates = {max_d:.4f}% (< 1%); fitted rates kept.
- Note (2026-09-27): the Cell 4b-2 brute-force ordered previous pair events by ts only; the LAG uses (ts, id). {n_groups:,} (src,dst,minute) groups have >1 transfer ({n_rows:,} rows); the 5 sample rows did not hit one. Feature correct; check weaker than claimed. 4c/4f checks use (ts, id).
- Note (2026-09-27): sum cold-start is inconsistent between 4a (0) and 4c (NULL) until 4f, which recomputes the no-history flag for sender sums via ASOF (#9d point 4).
- Note (2026-09-27): Cell 4c done, 8 receiver features, 25/34. dst_out_cnt_24h = receiver's own outbound activity (mixed perspective, intentional, #9d point 7).
"""
p = Path("../reports/pilot_log.md")
t = p.read_text(encoding="utf-8")
assert "#9d registered before Cell 4c" not in t, "notes already present"
with open(p, "a", encoding="utf-8") as f:
    f.write(TEXT)
print("written")

written
